Caitlin Kontaridis
Assignment 2

In [0]:
from pyspark.sql.functions import col,floor,lit, datediff,current_date,avg, when, min, max, year, sum, variance, expr
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
from pyspark.sql.types import IntegerType, DoubleType

In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True)

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:
pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
w_finish = spark.read.csv("/Volumes/gr5069/raw/f1_data/2022_driver_standings.csv", header=True)
w_finish = Window.partitionBy("raceId").orderBy("positionOrder")
w_pit = Window.partitionBy("raceId").orderBy("avg_duration")

Question 1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

Logic 
- I grouped the pit stop data by race and driver to calculate the average time a driver spent in pit stops during each race. Then, aggregate again at the race level to find the fastest (min) and slowest (max) pit stop times.

In [0]:
# Average pit stop / driver / race
avg_pit = pit_stops.groupBy("raceId", "driverId") \
  .agg(avg("milliseconds").alias("avg_pit_time"))

# Min and Max
race_pit_stats = pit_stops.groupBy("raceId") \
  .agg(
    min("milliseconds").alias("fastest_pit"),
    max("milliseconds").alias("slowest_pit")
  )

#Join
q1 = avg_pit.join(race_pit_stats, "raceId")
display(q1)

Explanation:
- groupBy("raceId", "driverId") - groups data per race and driver
- avg("milliseconds") - average pit stop time for each driver
- min() and max() - identifies the fastest/slowest pit stops per race
- join() - combines driver averages with race-level statistics

Alternative approach:
- Use Window Functions to compute race level stats without a second aggregation
- window_race = Window.partionBy("raceId")
pit_alt = pit_stops.withColumn("fastes_pit", min("milliseconds"),over(window_race))\
.withColumn("slowest_pit", max("milliseconds").over(window_race))

Question 2: Rank order by finishing position the average time spent at the pit stop in each race.

Logic:
- After computing the averages, the next step is to rank drivers within each race based on their average pit stop time (fastest to slowest).

In [0]:
# Time to rank
window_spec = Window.partitionBy("raceId").orderBy("avg_pit_time")
ranked_pit = avg_pit.withColumn("pit_rank", rank().over(window_spec))
display(ranked_pit)

Explanation:
- Window.partitionBy("raceId") - ranking within each race
- orderBy("avg_pit_time") - sorts drivers by pit time
- rank() - assigns rank

Alternatives:
- dense_rank() if we don't want gaps

Question 3:  Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

Logic:
- To fill missing code, I used the known driver names and tied it to 3 letters abbreviations. 

In [0]:
drivers_filled = drivers.withColumn(
    "code",
    when(col("code").isNull() & (col("surname") == "Alonso"), "ALO")
    .when(col("code").isNull() & (col("surname") == "Raikkonen"), "RAI")
    .when(col("code").isNull() & (col("surname") == "Rosberg"), "ROS")
    .when(col("code").isNull() & (col("surname") == "Hamilton"), "HAM")
    .otherwise(col("code"))
  )
display(drivers_filled)

Explanation:
- when or otherwise - conditional logic
- isNull() - finds missing driver codes
- updates missing values w/o overwriting existing ones

Alternatives:
- Dictionary Mapping
- mapping = {"Alonso":"ALO, "Hamilton": "HAM"}

Question 4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age"

Logic:
- Age is the differences between the race date and driver's date of birth. Then, this will find the youngest and oldest drivers using aggregation.

In [0]:
# Join drivers and races
df = results.join(drivers, "driverId").join(races, "raceId")

# Age
df_age = df.withColumn(
  "age",
  year(col("date")) - year(col("dob"))
)

# Youngest and oldest drivers
age_stats = df_age.groupBy("raceId").agg(
  min("age").alias("youngest"),
  max("age").alias("oldest")
)
display(age_stats)

Explanation:
- join() - combine driver and race data
- year(date) - finds age
- min and max - finds oldest and youngest

Alternatives:
- months_between()
- df_age = df.withColumn("age", months_between(col("date"), col("dob"))/12)

Question 5: At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

Logic:
- Each race, this will count how many times a driver finished in 1st, 2nd or 3rd place using cumulative sums

In [0]:
window_spec = Window.partitionBy("driverId").orderBy("raceId")
df_podium = results
df_podium = df_podium \
    .withColumn("win", when(col("position") == 1, 1).otherwise(0))\
    .withColumn("podium", when(col("position") <= 3, 1).otherwise(0))\
    .withColumn("second", when(col("position") == 2, 1).otherwise(0))\
    .withColumn("third", when(col("position") == 3, 1).otherwise(0))
df_podium = df_podium \
    .withColumn("total_wins", sum("win").over(window_spec)) \
    .withColumn("total_seconds", sum("second").over(window_spec))\
    .withColumn("total_thirds", sum("third").over(window_spec))\
    .withColumn("total_podiums", sum("podium").over(window_spec))
display(results)

Code Explanation:
- when() - creats columns for podium positions
- Window - cumalative counting over races
- sum().over() - running totals


Alternatives:
- Group instead of cumalative if per race is needed

Question 6: 
Which team has the most consistent finishing performance?

Logic:
- Grouping results by team and calculated the variance of finishing positions. Lower variance show more consistent performance.

In [0]:
df_clean = df.withColumn(
  "position",
  expr("try_cast(position as double)")
)
consistency = df_clean.groupBy("constructorId") \
  .agg(variance("position").alias("position_variance")) \
  .orderBy("position_variance")
display(consistency)

Explanation:
- variance() - measures spread of finishing positions
- lower values = more consistency
- orderBy() - ranks constructors (teams) by consistency

In [0]:
pit_stops.write.csv('/Volumes/gr5069/cmk2240/hw2/lap_times_output.csv')

In [0]:
!mkdir -p /tmp/gitconfig
!git config --global --add safe.directory /Workspace/Users/cmk2240@columbia.edu/testrepo-2026/src/HW2_Kontaridis

In [0]:
import os
os.environ["GIT_CONFIG_GLOBAL"] = "/tmp/gitconfig/.gitconfig"

In [0]:
!git config --global --add safe.directory "*"

In [0]:
!git config --global user.email "cmk2240@columbia.edu"
!git config --global user.name "Caitlin Kontaridis"


In [0]:
!git remote add origin https://github.com/QMSS-GR5069-Spring2026/take-home-exercise-2-cmk2240-3.git

In [0]:
!git add . 

In [0]:
!git status